# Read data (all 12 csv files)

nyc_taxi csv files are expected to be in the `data/` directory.

In [1]:
import dataset                          # Data loader
import pandas as pd                     # For the data manipulation
import numpy as np                      # For the numerical operations
import matplotlib.pyplot as plt         # Plots
import seaborn as sns                   # This is for the plots to look better

from xgboost import XGBRegressor        # Make sure to install pip install xgboost
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Dataset metadata

In [2]:
# Pulling total data is 4M - things get buggy;
# Using month 1 & 2 for now
df = dataset.raw(month_start=1, month_end=2)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (5972150, 20)

Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'month']

First 3 rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,month
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0,1
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,1
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0,1


## The data contains ALL NYC boroughs but we only care about Manhattan thus
###  We use the taxi zone lookup to filter down to Manhattan only


In [3]:
zones = pd.read_csv("../../data/taxi_zone_lookup.csv")

print(zones.head())
print("\nBoroughs available:", zones["Borough"].unique())

# We are getting just the Manhattan zone IDs
manhattan_ids = zones[zones["Borough"] == "Manhattan"]["LocationID"].tolist()
print(f"\nNumber of Manhattan zones: {len(manhattan_ids)}")

   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone

Boroughs available: ['EWR' 'Queens' 'Bronx' 'Manhattan' 'Staten Island' 'Brooklyn' 'Unknown'
 nan]

Number of Manhattan zones: 69


In [4]:
# We are keeping only Manhattan pickups
df = df[df["PULocationID"].isin(manhattan_ids)]

# Removing rows with invalid fares (negative or zero fare amounts don't make sense)
df = df[df["fare_amount"] > 0]

# Remove rows with invalid trip distances
df = df[df["trip_distance"] > 0]

# Remove rows where pickup is after dropoff (data error)
df = df[df["tpep_pickup_datetime"] < df["tpep_dropoff_datetime"]]

# Remove any rows with missing pickup times
df = df.dropna(subset=["tpep_pickup_datetime"])

print("Shape after cleaning:", df.shape)
print("\nAny nulls remaining?")
print(df.isnull().sum())

Shape after cleaning: (5198008, 20)

Any nulls remaining?
VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          241199
trip_distance                 0
RatecodeID               241199
store_and_fwd_flag       241199
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     241199
Airport_fee              241199
month                         0
dtype: int64
